# Transformer de panel: búsqueda automática de hiperparámetros (AutoML con Optuna)

Este notebook busca automáticamente la mejor configuración del **Transformer de panel**
(`PanelTransformerModel`, `src/panel_transformer.py`) para responder la pregunta que importa para
operar: **¿la acción va a subir o a bajar en las próximas 5 ruedas?** (comprar o vender).

**El modelo**

- Mira **todas las acciones del panel líder del mismo día juntas**: un encoder temporal por acción
  y atención entre acciones (`capas_cruzadas`), con features normalizadas por volatilidad
  (`src/panel_features.py`).
- Trabaja **en dólares CCL** (`moneda="usd"`): "sube" significa que **le gana al dólar**. En pesos,
  buena parte de cada suba es inflación y devaluación; en dólares queda la parte que importa para
  decidir una inversión.
- Usa `target="absoluto"`: el retorno a 5 ruedas en unidades de volatilidad, **sin restarle la media
  del día**. El target por defecto (`"sin_media"`) solo dice si una acción rinde más o menos que el
  resto, no si sube o baja.

**Cómo se evalúa**

1. **Datos**: snapshot local de la simulación histórica (`simulacion/snapshot`), con los precios
   limpios de splits sin ajustar (`src/simulacion/limpieza.py`). No depende de la red.
2. **Validación walk-forward**: para cada periodo se entrena con los datos *anteriores* a su inicio
   y se predice cada rueda del periodo sin reentrenar (`predecir_periodo`, igual que en
   `src/simulacion/busqueda.py`). El modelo nunca ve el futuro.
3. **AutoML**: Optuna (búsqueda bayesiana TPE) elige hiperparámetros con los **7 periodos de
   2022-2025**. Los trials que van claramente peor que la mediana se cortan antes de terminar
   (*pruning*).
4. **Robustez**: los mejores trials se re-evalúan con otras semillas, porque el ruido entre semillas
   es grande y el mejor trial suele estar "inflado" por suerte.
5. **Confirmación**: los **2 periodos de 2026** quedan reservados y se miran **una sola vez** al
   final, con la configuración ya elegida.
6. **Modelo final**: se entrena con todos los datos y se muestra qué dice hoy para cada acción.

**Métrica: balanced accuracy de la dirección** (`acierto_balanceado` en `src/trend_common.py`)

Es el promedio entre el *acierto en subas* (de las veces que la acción subió, cuántas dijo "sube") y
el *acierto en bajas*. Un modelo que dice siempre "sube" (o siempre "baja") saca exactamente 50%, el
azar, aunque su accuracy común sea alta en un periodo alcista. Además se reporta la accuracy común,
el IC (si ordena bien las acciones entre sí) y cuánto rinden las que dijo "sube" contra las que dijo
"baja".

**Requisitos**: correr en `PC-Escritorio` con el kernel de Python del host (tiene torch con CUDA),
desde `api_ml/` o `api_ml/notebooks/`, con `optuna` instalado (`pip install optuna`, está en
`requirements-dev.txt`) y el snapshot descargado (`python -m src.simulacion.snapshot`). Cada trial
tarda 2-3 minutos en la GPU (7 entrenamientos); la búsqueda completa, unas horas. **Se puede cortar
y retomar**: el estudio queda en `simulacion/automl/optuna.db` y las predicciones de cada periodo,
en `simulacion/automl/predicciones/`.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Raiz del proyecto (api_ml/): el snapshot y los resultados usan rutas relativas a ella.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "panel_transformer.py").exists()),
    Path.cwd(),
)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import contextlib
import hashlib
import io
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch

from src.black_litterman.black_litterman import PANEL_LIDER_TICKERS
from src.config import settings
from src.panel_datos import ccl_en, en_dolares
from src.panel_transformer import PanelTransformerConfig, PanelTransformerModel
from src.simulacion.busqueda import (
    PERIODOS_CONFIRMACION,
    PERIODOS_ELECCION,
    cargar_snapshot,
    proveedor_snapshot,
)
from src.simulacion.entrenamiento import histories_at
from src.simulacion.experimento import predecir_periodo
from src.trend_common import acierto_balanceado, derive_trend_output, ic_por_fecha, senal

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
warnings.filterwarnings("ignore", message="Some regimes")  # importancias con pocos trials
warnings.filterwarnings("ignore", message="Attempting to set identical")  # slice de un solo valor
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
N_TRIALS = 60  # trials de la busqueda (cuenta los ya hechos si se retoma)
TIMEOUT_HORAS = 3.0  # corta la busqueda aunque no se llegue a N_TRIALS
TOP_K = 3  # mejores trials que se re-evaluan con mas semillas
SEMILLA = 42  # semilla de los trials (la misma que usa busqueda.py)
SEMILLAS_EXTRA = [43, 44]  # semillas de la re-evaluacion de robustez
NOMBRE_ESTUDIO = "panel-transformer-direccion-usd"
HORIZON = PanelTransformerConfig.horizonte
BANDA_NEUTRAL = 0.01  # |retorno esperado| por debajo => senal "neutral" (como en la web)

DIR_SALIDA = Path("simulacion/automl")  # gitignoreado, como el resto de simulacion/
DIR_SALIDA.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{DIR_SALIDA / 'optuna.db'}"

dispositivo = torch.cuda.get_device_name() if DISPOSITIVO == "cuda" else "cpu"
print(f"Dispositivo: {dispositivo}")

## 2. Datos

Snapshot con precios limpios y la serie del dólar CCL. El modelo se entrena y predice sobre las
acciones del **panel líder**. El CCL de cada rueda es el último publicado hasta esa fecha
(`ccl_en`), igual que adentro del modelo: nunca se usa un dato posterior.

In [ ]:
snapshot = cargar_snapshot("simulacion/snapshot", limpiar=True)
proveedor = proveedor_snapshot(snapshot)
ruedas = snapshot.trading_days()
CCL = snapshot.ccl_hasta(ruedas[-1])
print(f"{len(snapshot.tickers)} tickers, ruedas {ruedas[0].date()} -> {ruedas[-1].date()}")
print(f"CCL: {CCL.index[0].date()} -> {CCL.index[-1].date()}, ultimo {CCL.iloc[-1]:,.0f}")

pd.DataFrame(
    [{"uso": "eleccion (busqueda)", "desde": d, "hasta": h} for d, h in PERIODOS_ELECCION]
    + [{"uso": "confirmacion (una vez)", "desde": d, "hasta": h} for d, h in PERIODOS_CONFIRMACION]
)

`a_dolares` convierte el retorno real a 5 ruedas (que `predecir_periodo` calcula en pesos) a
dólares: le resta la variación del CCL entre la fecha de la predicción y 5 ruedas después.

In [ ]:
def variacion_ccl(df: pd.DataFrame, ruedas_adelante: int) -> np.ndarray:
    # log(CCL a `ruedas_adelante` ruedas del ticker / CCL en la fecha), fila por fila.
    salida = np.full(len(df), np.nan)
    for symbol, g in df.groupby("symbol"):
        idx = snapshot.prices[symbol].index
        pos = idx.get_indexer(pd.to_datetime(g["fecha"]))
        destino = pos + ruedas_adelante
        ok = (pos >= 0) & (destino >= 0) & (destino < len(idx))
        desde = ccl_en(CCL, pd.DatetimeIndex(idx[pos[ok]])).to_numpy()
        hasta = ccl_en(CCL, pd.DatetimeIndex(idx[destino[ok]])).to_numpy()
        valores = np.full(len(g), np.nan)
        valores[ok] = np.log(hasta / desde)
        salida[g.index.to_numpy()] = valores
    return salida


def a_dolares(df: pd.DataFrame) -> pd.DataFrame:
    # Retorno real a HORIZON ruedas en dolares CCL (`real`); el de pesos queda en `real_ars`.
    df = df.reset_index(drop=True).rename(columns={"real": "real_ars"})
    df["real"] = df["real_ars"] - variacion_ccl(df, HORIZON)
    return df.dropna(subset=["real"])

### Pesos vs. dólares

Fracción de casos (acción del panel × rueda) en los que el precio a 5 ruedas terminó más alto, en
pesos y en dólares. En pesos, la inflación y la devaluación de cada periodo empujan los precios
hacia arriba; en dólares solo cuenta si la acción le ganó al dólar. En ningún caso el reparto es
50/50, por eso la accuracy común engaña y se usa la balanced accuracy.

In [ ]:
def casos_periodo(desde: str, hasta: str) -> pd.DataFrame:
    # Retorno real a HORIZON ruedas de cada accion del panel, por rueda del periodo.
    partes = []
    for symbol in PANEL_LIDER_TICKERS:
        if symbol in snapshot.prices:
            close = snapshot.prices[symbol]["close"]
            real = np.log(close.shift(-HORIZON) / close).loc[desde:hasta].dropna()
            fechas = real.index.strftime("%Y-%m-%d")
            partes.append(pd.DataFrame({"fecha": fechas, "symbol": symbol, "real": real.values}))
    return a_dolares(pd.concat(partes))


todos_periodos = PERIODOS_ELECCION + PERIODOS_CONFIRMACION
subas = pd.DataFrame(
    {
        d[:7]: {"pesos": (c["real_ars"] > 0).mean(), "dolares": (c["real"] > 0).mean()}
        for d, h in todos_periodos
        for c in [casos_periodo(d, h)]
    }
).T

ax = (subas * 100).plot.bar(figsize=(10, 3.5), rot=0)
ax.axhline(50, color="black", lw=1, ls="--")
ax.axvline(len(PERIODOS_ELECCION) - 0.5, color="gray", lw=1)
ax.set_ylabel("% de casos que subieron")
ax.set_title("Casos que subieron a 5 ruedas (a la derecha de la linea gris: confirmacion)")
ax.set_ylim(0, 100)
plt.show()

## 3. Métricas y evaluación walk-forward

`evaluar` entrena un modelo al inicio de cada periodo (con datos hasta la rueda anterior) y predice
todas las ruedas del periodo. Devuelve una fila por (fecha, acción) con el retorno log predicho en
dólares (`pred`), la señal de la web (`signal`: alza / baja / neutral con la banda de ±1%) y el
retorno real a 5 ruedas en dólares (`real`). Las predicciones se guardan en disco (por
configuración, semilla y periodo) y no se recalculan: si se cambia el código del modelo, borrar
`simulacion/automl/predicciones/`.

La decisión de compra/venta es el signo de la predicción: `pred > 0` → comprar, si no → vender.

In [ ]:
def evaluar(nombre: str, fabrica, periodos: list[tuple[str, str]]) -> pd.DataFrame:
    # Predicciones walk-forward de `fabrica` en cada periodo (cacheadas por nombre y periodo).
    partes = []
    for desde, hasta in periodos:
        archivo = DIR_SALIDA / "predicciones" / f"{nombre}__{desde}.csv"
        if archivo.exists():
            df = pd.read_csv(archivo)
        else:
            df = predecir_periodo(snapshot, fabrica, desde, hasta, PANEL_LIDER_TICKERS)
            archivo.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(archivo, index=False)
        partes.append(df.assign(periodo=desde[:7]))
    return a_dolares(pd.concat(partes, ignore_index=True))


def metricas_direccion(df: pd.DataFrame) -> dict:
    # Metricas de un conjunto de predicciones (retornos log a HORIZON ruedas, en dolares).
    subio = df["real"] > 0
    compra = df["pred"] > 0
    alza, baja = df["signal"] == "alza", df["signal"] == "baja"
    ret_compra = df["real"][compra].mean() * 100 if compra.any() else np.nan
    ret_venta = df["real"][~compra].mean() * 100 if (~compra).any() else np.nan
    return {
        "bal_acc": acierto_balanceado(df["pred"].to_numpy(), df["real"].to_numpy()),
        "accuracy": (compra == subio).mean(),
        "acierto_suba": compra[subio].mean() if subio.any() else np.nan,
        "acierto_baja": (~compra[~subio]).mean() if (~subio).any() else np.nan,
        "pct_dice_sube": compra.mean(),
        "pct_subio": subio.mean(),
        # Retorno log medio real (%, en USD a HORIZON ruedas) de las que dijo comprar / vender.
        "ret_compra": ret_compra,
        "ret_venta": ret_venta,
        "ventaja": ret_compra - ret_venta,
        "ic": ic_por_fecha(df["pred"].to_numpy(), df["real"].to_numpy(), df["fecha"].to_numpy()),
        # Senal de la web (banda neutral de +-1%): cuantas veces se juega y cuanto acierta.
        "pct_senal": (alza | baja).mean(),
        "precision_alza": (df["real"][alza] > 0).mean() if alza.any() else np.nan,
        "precision_baja": (df["real"][baja] < 0).mean() if baja.any() else np.nan,
    }


def por_periodo(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({p: metricas_direccion(g) for p, g in df.groupby("periodo")}).T


def resumen(df: pd.DataFrame) -> dict:
    # Promedio entre periodos (cada periodo pesa igual) y cuantos superan el azar.
    tabla = por_periodo(df)
    return {**tabla.mean().to_dict(), "periodos_sobre_50": int((tabla["bal_acc"] > 0.5).sum())}


COLUMNAS = [
    "bal_acc", "periodos_sobre_50", "accuracy", "acierto_suba", "acierto_baja", "pct_dice_sube",
    "ventaja", "ic", "pct_senal", "precision_alza", "precision_baja",
]  # fmt: skip


def comparar(resultados: dict[str, pd.DataFrame]) -> pd.DataFrame:
    tabla = pd.DataFrame({n: resumen(df) for n, df in resultados.items()}).T[COLUMNAS]
    return tabla.sort_values("bal_acc", ascending=False).astype(float).round(4)


def promediar(dfs: list[pd.DataFrame]) -> pd.DataFrame:
    # Ensamble: promedio de las predicciones de varias corridas (p. ej. semillas).
    df = (
        pd.concat(dfs)
        .groupby(["periodo", "fecha", "symbol"], as_index=False)
        .agg(pred=("pred", "mean"), real=("real", "first"), real_ars=("real_ars", "first"))
    )
    df["signal"] = [senal(float(np.expm1(p)), BANDA_NEUTRAL) for p in df["pred"]]
    return df


def clave(config: dict, seed: int = SEMILLA) -> str:
    # Nombre de cache de una configuracion + semilla (no depende del numero de trial).
    texto = json.dumps({**config, "seed": seed}, sort_keys=True)
    return "panel_" + hashlib.md5(texto.encode()).hexdigest()[:12]


def fabrica_panel(config: dict, seed: int = SEMILLA):
    return lambda: PanelTransformerModel(
        PanelTransformerConfig(**{**config, "seed": seed}),
        proveedor=proveedor,
        dispositivo=DISPOSITIVO,
    )

## 4. Referencias

Contra qué se compara cualquier configuración:

- **siempre_sube**: dice "sube" para todo. Balanced accuracy = 50% por definición; su accuracy
  común muestra cuánto se puede "acertar" sin saber nada.
- **reversion_usd**: regla sin entrenamiento, predice que el retorno en dólares de las últimas 5
  ruedas se revierte (en la simulación, la reversión fue la mejor señal simple para ordenar
  acciones).

In [ ]:
class ReversionDolares:
    # Predice que el retorno en dolares de las ultimas HORIZON ruedas se revierte.

    def fit(self, histories: dict[str, pd.DataFrame]) -> dict:
        return {}

    def predict_panel(self, dfs: dict[str, pd.DataFrame]) -> dict[str, dict]:
        resultados = {}
        for symbol, df in dfs.items():
            close = en_dolares(df.iloc[-HORIZON - 1 :], CCL)["close"].to_numpy()
            if len(close) <= HORIZON or not np.isfinite(close).all() or (close <= 0).any():
                continue
            pasado = float(np.log(close[-1] / close[0]))
            resultados[symbol] = derive_trend_output(df, -pasado, HORIZON, BANDA_NEUTRAL)
        return resultados


reversion = evaluar("reversion_usd", ReversionDolares, PERIODOS_ELECCION)
referencias_eleccion = {
    "reversion_usd": reversion,
    "siempre_sube": reversion.assign(pred=0.01, signal="alza"),
}
comparar(referencias_eleccion)

## 5. Espacio de búsqueda

Hiperparámetros de `PanelTransformerConfig` que explora Optuna. Quedan fijos `target="absoluto"`,
`moneda="usd"` y `universo="panel"`.

| Grupo | Parámetros |
|---|---|
| Arquitectura | `ventana` (ruedas que mira cada acción), `d_model`, `nhead`, `dim_feedforward`, `capas_temporales`, `capas_cruzadas` (atención entre acciones; 0 = cada acción sola), `dropout`, `emb_sector`, `emb_ticker` |
| Datos | `macro` (token con el contexto macro del día: CCL, brecha, riesgo país, tasas), `vida_media_anios` (cuánto pesan los datos viejos) |
| Pérdida | `perdida` (`huber` o `combinada` = huber − λ·IC), `lambda_ic`, `huber_delta` |
| Early stopping | `seleccion` (`direccion`: balanced accuracy en validación; `ic`: ordenar bien), `paciencia` |
| Optimización | `learning_rate`, `weight_decay`, `batch_dias`, `warmup_epocas` |

Arranca con dos configuraciones conocidas: la default del modelo y la mejor de la fase 4 del plan
(ventana 1 y regularización fuerte, IC 0,059), las dos con target absoluto.

In [ ]:
FIJOS = {"target": "absoluto", "moneda": "usd", "universo": "panel"}


def sugerir_config(trial: optuna.Trial) -> dict:
    d_model = trial.suggest_categorical("d_model", [16, 32, 64])
    perdida = trial.suggest_categorical("perdida", ["huber", "combinada"])
    config = {
        **FIJOS,
        "ventana": trial.suggest_categorical("ventana", [1, 5, 10, 20, 40]),
        "d_model": d_model,
        "nhead": trial.suggest_categorical("nhead", [2, 4]),
        "dim_feedforward": d_model * trial.suggest_categorical("ff_mult", [2, 4]),
        "capas_temporales": trial.suggest_int("capas_temporales", 1, 2),
        "capas_cruzadas": trial.suggest_int("capas_cruzadas", 0, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "emb_sector": trial.suggest_categorical("emb_sector", [True, False]),
        "emb_ticker": trial.suggest_categorical("emb_ticker", [False, True]),
        "macro": trial.suggest_categorical("macro", [False, True]),
        "vida_media_anios": trial.suggest_categorical("vida_media_anios", [None, 1.0, 2.0, 4.0]),
        "perdida": perdida,
        "huber_delta": trial.suggest_categorical("huber_delta", [0.5, 1.0, 2.0]),
        "seleccion": trial.suggest_categorical("seleccion", ["direccion", "ic"]),
        "paciencia": trial.suggest_int("paciencia", 5, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-4, 0.3, log=True),
        "batch_dias": trial.suggest_categorical("batch_dias", [8, 16, 32]),
        "warmup_epocas": trial.suggest_int("warmup_epocas", 0, 5),
    }
    if perdida == "combinada":
        config["lambda_ic"] = trial.suggest_float("lambda_ic", 0.1, 2.0, log=True)
    return config


_d = PanelTransformerConfig()
PARAMS_DEFAULT = {
    "d_model": _d.d_model, "perdida": _d.perdida, "ventana": _d.ventana, "nhead": _d.nhead,
    "ff_mult": _d.dim_feedforward // _d.d_model, "capas_temporales": _d.capas_temporales,
    "capas_cruzadas": _d.capas_cruzadas, "dropout": _d.dropout, "emb_sector": _d.emb_sector,
    "emb_ticker": _d.emb_ticker, "macro": _d.macro, "vida_media_anios": _d.vida_media_anios,
    "huber_delta": _d.huber_delta, "seleccion": "direccion", "paciencia": _d.paciencia,
    "learning_rate": _d.learning_rate, "weight_decay": _d.weight_decay,
    "batch_dias": _d.batch_dias, "warmup_epocas": _d.warmup_epocas, "lambda_ic": _d.lambda_ic,
}  # fmt: skip
# Mejor configuracion de la fase 4 (pt_w1_reg_mas: IC 0,059 con target "sin_media").
PARAMS_FASE4 = PARAMS_DEFAULT | {
    "ventana": 1, "dropout": 0.3, "weight_decay": 0.3, "learning_rate": 1e-4
}  # fmt: skip

## 6. Objetivo

Cada trial entrena y predice los 7 periodos de elección, uno por vez. Después de cada periodo le
informa a Optuna la balanced accuracy promedio hasta ahí: si va claramente peor que la mediana de los
trials anteriores en ese mismo punto, se corta (*pruning*) y se ahorra el resto. El valor final es la
balanced accuracy promedio de los 7 periodos.

In [ ]:
def objetivo(trial: optuna.Trial) -> float:
    config = sugerir_config(trial)
    trial.set_user_attr("config", config)
    tablas = []
    for paso, periodo in enumerate(PERIODOS_ELECCION):
        df = evaluar(clave(config), fabrica_panel(config), [periodo])
        tablas.append(por_periodo(df))
        trial.report(float(pd.concat(tablas)["bal_acc"].mean()), paso)
        if trial.should_prune():
            raise optuna.TrialPruned()
    tabla = pd.concat(tablas)
    for metrica in ["accuracy", "acierto_suba", "acierto_baja", "pct_dice_sube", "ventaja", "ic"]:
        trial.set_user_attr(metrica, float(tabla[metrica].mean()))
    trial.set_user_attr("periodos_sobre_50", int((tabla["bal_acc"] > 0.5).sum()))
    return float(tabla["bal_acc"].mean())


def informar(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    valor = f"{trial.value:.4f}" if trial.value is not None else "  -   "
    completos = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    mejor = f"{study.best_value:.4f} (trial {study.best_trial.number})" if completos else "-"
    estado = f"trial {trial.number:>3} {trial.state.name:<8}"
    print(f"{estado} bal_acc={valor}  mejor={mejor}", flush=True)

## 7. Búsqueda

TPE multivariado (modela las interacciones entre hiperparámetros) con 10 trials aleatorios
iniciales. El pruner no corta nada en los primeros 10 trials ni antes del tercer periodo.

Se puede interrumpir (botón de stop del kernel) y volver a correr la celda: retoma desde el estudio
guardado.

In [ ]:
study = optuna.create_study(
    study_name=NOMBRE_ESTUDIO,
    storage=STORAGE,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEMILLA, multivariate=True, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
    load_if_exists=True,
)
if not study.trials:
    study.enqueue_trial(PARAMS_DEFAULT)
    study.enqueue_trial(PARAMS_FASE4)

hechos = sum(t.state.is_finished() for t in study.trials)
if hechos < N_TRIALS:
    print(f"{hechos} trials hechos, faltan {N_TRIALS - hechos}")
    study.optimize(
        objetivo,
        n_trials=N_TRIALS - hechos,
        timeout=TIMEOUT_HORAS * 3600,
        callbacks=[informar],
        gc_after_trial=True,
        # Un trial que falla (p. ej. sin memoria en la GPU) no corta la busqueda.
        catch=(Exception,),
    )
print(f"Mejor trial: {study.best_trial.number}, bal_acc={study.best_value:.4f}")

## 8. Resultados de la búsqueda

In [ ]:
print(pd.Series([t.state.name for t in study.trials]).value_counts().to_string(), "\n")

trials = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
trials = trials[trials["state"] == "COMPLETE"].sort_values("value", ascending=False)
trials = (
    trials.drop(columns=["state", "user_attrs_config"])
    .rename(columns=lambda c: c.removeprefix("params_").removeprefix("user_attrs_"))
    .rename(columns={"value": "bal_acc"})
)
trials.head(10).round(4)

In [ ]:
from optuna.visualization import matplotlib as vis

vis.plot_optimization_history(study)
plt.title("Balanced accuracy por trial")
plt.show()

**Importancia de los hiperparámetros** (fANOVA): cuánto de la variación de la balanced accuracy
entre trials explica cada parámetro. Ayuda a ver qué conviene fijar y qué seguir explorando.

In [ ]:
vis.plot_param_importances(study)
plt.show()

In [ ]:
importancias = optuna.importance.get_param_importances(study)
vis.plot_slice(study, params=list(importancias)[:6])
plt.show()

## 9. Robustez: re-evaluación con otras semillas

El mejor trial de una búsqueda casi siempre está inflado: entre muchas configuraciones, alguna sale
bien por la semilla. Los `TOP_K` mejores se re-entrenan con `SEMILLAS_EXTRA` y se elige por el
**promedio entre semillas**. También se mide el **ensamble** (promedio de las predicciones de las
semillas, lo que hace `n_semillas` en `PanelTransformerConfig`).

In [ ]:
completos = sorted(
    (t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
    key=lambda t: t.value,
    reverse=True,
)[:TOP_K]

filas, predicciones = [], {}
for t in completos:
    config = t.user_attrs["config"]
    dfs = [evaluar(clave(config), fabrica_panel(config), PERIODOS_ELECCION)]
    for seed in SEMILLAS_EXTRA:
        print(f"trial {t.number}: semilla {seed}", flush=True)
        dfs.append(evaluar(clave(config, seed), fabrica_panel(config, seed), PERIODOS_ELECCION))
    predicciones[t.number] = dfs
    por_semilla = [resumen(df)["bal_acc"] for df in dfs]
    ensamble = resumen(promediar(dfs))
    filas.append(
        {
            "trial": t.number,
            "bal_acc_busqueda": t.value,
            "bal_acc_medio": np.mean(por_semilla),
            "desvio_semillas": np.std(por_semilla),
            "bal_acc_ensamble": ensamble["bal_acc"],
            "periodos_sobre_50_ens": ensamble["periodos_sobre_50"],
            "ventaja_ens": ensamble["ventaja"],
        }
    )

robustez = pd.DataFrame(filas).set_index("trial").sort_values("bal_acc_medio", ascending=False)
robustez.round(4)

In [ ]:
MEJOR = int(robustez.index[0])
fila = robustez.loc[MEJOR]
USAR_ENSAMBLE = bool(fila["bal_acc_ensamble"] > fila["bal_acc_medio"])
SEMILLAS_FINAL = [SEMILLA, *SEMILLAS_EXTRA] if USAR_ENSAMBLE else [SEMILLA]
CONFIG_ELEGIDA = study.trials[MEJOR].user_attrs["config"]

modo = f"ensamble de {len(SEMILLAS_FINAL)} semillas" if USAR_ENSAMBLE else "una semilla"
print(f"Elegido: trial {MEJOR} ({modo})")
pd.Series(CONFIG_ELEGIDA, name="valor").to_frame()

### Elegido vs. referencias en 2022-2025, periodo por periodo

In [ ]:
elegido_eleccion = promediar(predicciones[MEJOR]) if USAR_ENSAMBLE else predicciones[MEJOR][0]
eleccion = {"automl": elegido_eleccion, **referencias_eleccion}
display(comparar(eleccion))

bal_por_periodo = pd.DataFrame({n: por_periodo(df)["bal_acc"] for n, df in eleccion.items()})
ax = (bal_por_periodo * 100).plot.bar(figsize=(10, 4), rot=0)
ax.axhline(50, color="black", lw=1, ls="--")
ax.set_ylabel("balanced accuracy (%)")
ax.set_ylim(35, 65)
ax.set_title("Balanced accuracy por periodo de eleccion")
plt.show()

## 10. Confirmación en 2026 (fuera de muestra)

Los periodos de 2026 no se usaron para nada hasta acá. **Mirar una sola vez**: si después de ver
este resultado se vuelve a cambiar la búsqueda, 2026 deja de ser una prueba limpia.

In [ ]:
dfs = [
    evaluar(clave(CONFIG_ELEGIDA, s), fabrica_panel(CONFIG_ELEGIDA, s), PERIODOS_CONFIRMACION)
    for s in SEMILLAS_FINAL
]
reversion_conf = evaluar("reversion_usd", ReversionDolares, PERIODOS_CONFIRMACION)
confirmacion = {
    "automl": promediar(dfs) if len(dfs) > 1 else dfs[0],
    "reversion_usd": reversion_conf,
    "siempre_sube": reversion_conf.assign(pred=0.01, signal="alza"),
}
display(comparar(confirmacion))
pd.DataFrame({n: por_periodo(df)["bal_acc"] for n, df in confirmacion.items()}).round(4)

## 11. ¿Sirve para comprar y vender?

**Matriz de confusión** en 2026: de las veces que dijo comprar/vender, qué pasó (en dólares).

In [ ]:
df = confirmacion["automl"]
matriz = pd.crosstab(
    np.where(df["pred"] > 0, "dijo sube (comprar)", "dijo baja (vender)"),
    np.where(df["real"] > 0, "subio", "bajo"),
    rownames=["dijo"],
    colnames=["paso"],
    normalize="index",
)
print("% de cada fila: de las veces que dijo X, cuantas subio / bajo (en USD)")
(matriz * 100).round(1)

**Señal de la web** (alza / baja / neutral con la banda de ±1%): cuántas veces se juega y con qué
precisión. Una señal aporta si su precisión supera la tasa base (fracción de casos que subieron,
para "alza"; que bajaron, para "baja").

In [ ]:
filas = []
for nombre, df in confirmacion.items():
    base = {"alza": (df["real"] > 0).mean(), "baja": (df["real"] < 0).mean()}
    for s in ["alza", "baja", "neutral"]:
        sel = df[df["signal"] == s]
        acierto = (sel["real"] > 0) if s == "alza" else (sel["real"] < 0)
        filas.append(
            {
                "modelo": nombre,
                "senal": s,
                "pct_casos": len(sel) / len(df),
                "precision": acierto.mean() if s != "neutral" and len(sel) else np.nan,
                "tasa_base": base.get(s),
                "ret_medio_usd_%": sel["real"].mean() * 100 if len(sel) else np.nan,
            }
        )
pd.DataFrame(filas).set_index(["modelo", "senal"]).astype(float).round(4)

**Estrategia simple**: cada 5 ruedas, comprar (en partes iguales) las acciones del panel que el
modelo dice que suben y quedarse en dólares con el resto; se compara con comprar todo el panel.
Valores en dólares CCL y sin comisiones ni impuestos, así que es una cota optimista: rotar mucho la
cartera tiene costo.

In [ ]:
def curva_estrategia(df: pd.DataFrame) -> pd.Series:
    # Valor en USD de rebalancear cada HORIZON ruedas comprando las que dice que suben.
    fechas = sorted(df["fecha"].unique())[::HORIZON]
    rets = {}
    for f in fechas:
        g = df[df["fecha"] == f]
        compra = g[g["pred"] > 0]
        rets[f] = compra["real"].mean() if len(compra) else 0.0  # afuera: en dolares, rinde 0
    # Valor de la cartera en cada fecha de compra (antes del retorno de esas HORIZON ruedas).
    return np.exp(pd.Series(rets).cumsum().shift(1, fill_value=0.0))


fig, axes = plt.subplots(1, len(PERIODOS_CONFIRMACION), figsize=(13, 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (desde, _) in zip(axes, PERIODOS_CONFIRMACION, strict=True):
    for nombre, estilo in [("automl", "-"), ("reversion_usd", "-"), ("siempre_sube", "--")]:
        df = confirmacion[nombre]
        curva = curva_estrategia(df[df["periodo"] == desde[:7]])
        etiqueta = "comprar todo" if nombre == "siempre_sube" else nombre
        ax.plot(pd.to_datetime(curva.index), curva.values, ls=estilo, label=etiqueta)
    ax.set_title(f"Desde {desde}")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("valor de USD 1")
axes[0].legend()
plt.tight_layout()
plt.show()

## 12. Modelo final y qué dice hoy

Se entrena la configuración elegida con todos los datos del snapshot (hasta la última rueda) y se
guarda en `simulacion/automl/panel_transformer_automl.pt` (redes, config y normalizadores; la
config sola también queda en `config_final.json`). Se carga con
`PanelTransformerModel.load(ruta, proveedor=...)`: el proveedor del CCL / la macro no se guarda,
se pasa al cargar. `PanelTransformerModel` no está en el registro de api_ml: **no reemplaza a
ningún modelo de producción**.

"Comprar" significa que espera que la acción **le gane al dólar CCL** en las próximas 5 ruedas
(retorno esperado > +1%), y "Vender", que pierda contra el dólar (< −1%). Una acción puede subir en
pesos y aun así ser "Vender".

Las predicciones son al cierre de la última rueda del snapshot; para tener datos de hoy, volver a
descargarlo con `python -m src.simulacion.snapshot` y re-correr el notebook (las predicciones
cacheadas siguen sirviendo).

In [ ]:
CONFIG_FINAL = {**CONFIG_ELEGIDA, "n_semillas": len(SEMILLAS_FINAL)}
(DIR_SALIDA / "config_final.json").write_text(json.dumps(CONFIG_FINAL, indent=2))

ultima = ruedas[-1]
historias = histories_at(snapshot, ultima, settings.history_days)
modelo_final = fabrica_panel(CONFIG_FINAL)()
with contextlib.redirect_stdout(io.StringIO()):
    metricas_val = modelo_final.fit(historias)
print("Metricas en validacion interna:", metricas_val)

RUTA_MODELO = DIR_SALIDA / "panel_transformer_automl.pt"
modelo_final.save(RUTA_MODELO)
print(f"Guardado en {RUTA_MODELO}")

In [ ]:
DECISION = {"alza": "Comprar", "baja": "Vender", "neutral": "Mantener"}

# Se predice con el modelo cargado del disco, como se usaria despues sin reentrenar.
modelo_cargado = PanelTransformerModel.load(RUTA_MODELO, proveedor=proveedor)
panel = {s: historias[s] for s in PANEL_LIDER_TICKERS if s in historias}
hoy = modelo_cargado.predict_panel(panel)
assert hoy == modelo_final.predict_panel(panel), "el modelo cargado no predice igual"
tabla_hoy = pd.DataFrame(
    {
        s: {
            "ultimo_cierre_ars": r["last_close"],
            "retorno_esperado_usd_%": (r["predicted_close"] / r["last_close"] - 1) * 100,
            "senal": r["signal"],
            "decision": DECISION[r["signal"]],
            "rsi": r["rsi"],
        }
        for s, r in hoy.items()
    }
).T.sort_values("retorno_esperado_usd_%", ascending=False)
print(f"Prediccion a {HORIZON} ruedas desde el cierre del {ultima.date()} (en dolares CCL)")
tabla_hoy

## Cómo leer los resultados

- Una balanced accuracy de **52-55% sostenida en varios periodos** ya es una señal útil en
  mercados; valores mayores al 60% en walk-forward son sospechosos (revisar si hay fuga de datos).
- La dirección en dólares depende mucho de **cómo se mueve todo el mercado** (y el CCL), no solo de
  cada acción: es más difícil de predecir que el orden entre acciones (el IC).
- El ruido entre semillas es grande: diferencias menores al `desvio_semillas` de la sección 9 son
  empate. El early stopping por dirección (`seleccion="direccion"`) es más ruidoso que por IC, con
  pocos días de validación puede quedarse con una época temprana por suerte.
- `periodos_sobre_50` importa tanto como el promedio: un modelo que gana mucho en un periodo y
  pierde en el resto no es confiable.
- `ventaja` > 0 significa que las que dijo "comprar" rindieron más (en dólares) que las que dijo
  "vender". Es lo que se traduce en plata, pero sin descontar comisiones.